In [32]:
import concurrent.futures
import glob
import gzip

import numpy as np
import pandas as pd


def process_base_file(base_file, mito_length):
    cur_base_data = pd.read_csv(gzip.open(base_file), header=None)

    # gather coverage for forward strand
    fwd_base_df = cur_base_data[[0, 1, 2]].pivot_table(index=1, columns=0)
    fwd_base_df.columns = [x[1] for x in fwd_base_df.columns.values]  # flatten weird multiindex after pivot
    fwd_base_df.index.name = None
    all_columns = list(range(1, mito_length + 1))
    fwd_base_df = fwd_base_df.reindex(columns=all_columns, fill_value=0)
    fwd_base_df = fwd_base_df.fillna(0).sort_index(axis=1)  # assume all nan are true zeroes

    # gather coverage for reverse strand
    rev_base_df = cur_base_data[[0, 1, 3]].pivot_table(index=1, columns=0)
    rev_base_df.columns = [x[1] for x in rev_base_df.columns.values]
    rev_base_df.index.name = None
    rev_base_df = rev_base_df.reindex(columns=all_columns, fill_value=0)
    rev_base_df = rev_base_df.fillna(0).sort_index(axis=1)

    return fwd_base_df, rev_base_df


def load_mgatk_output(output_dir, mito_length, sample_prefix):
    # assuming mgatk output naming convention
    base_files = [glob.glob(f"{output_dir}/{sample_prefix}.{nt}.txt.gz")[0] for nt in "ATCG"]

    base_coverage_dict = {}
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = {executor.submit(process_base_file, base_file, mito_length): nt for base_file, nt in zip(base_files, "ATCG")}
        for future in concurrent.futures.as_completed(futures):
            nt = futures[future]
            base_coverage_dict[nt] = future.result()

    return base_coverage_dict


def gather_possible_variants(base_coverage_dict, reference_file):
    # sum across cells and strands for each base and position
    aggregated_genotype = pd.DataFrame(np.zeros((4, mito_length)), index=list("ATCG"), columns=np.arange(1, mito_length + 1))
    for nt in base_coverage_dict:
        # sum across cells for each strand separately
        fwd_base_df, rev_base_df = base_coverage_dict[nt]
        fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

        # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
        masking = ~((fwd_base_sum > 0) & (rev_base_sum > 0))  # True if position not >0 for both strands
        fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

        # sum across strands
        aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum

    # make a reference set of tuples (pos, ref_base)
    ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]
    ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]
    ref_set = {(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters}
    ref_dict = dict(ref_set)

    # make an observed set of tuples which are nonzero
    non_zero_idx = np.where(aggregated_genotype > 0)
    non_zero_bases = [letters[i] for i in non_zero_idx[0]]
    non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]
    observed_set = list(zip(non_zero_pos, non_zero_bases))
    observed_set = {x for x in observed_set if x[0] not in ref_N_positions}  # disregard positions in ref with N

    # take difference between observed and reference
    variant_set = observed_set - ref_set
    variants = sorted([(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0])  # (pos, ref base, obs base)

    return variants


In [33]:
MGATK_OUT_DIR = "/home/liuc9/github/scMOCHA-data/data/GSE149689/targz/GSM4509027"
sample_prefix = "cell"
mito_length = int(16569)  # 16569
low_coverage_threshold = int(10)  # 10
mito_genome = "MT"  # chrM


In [34]:
output_dir = MGATK_OUT_DIR
[glob.glob(f"{output_dir}/{sample_prefix}.{nt}.txt.gz")[0] for nt in "ATCG"]

['/home/liuc9/github/scMOCHA-data/data/GSE149689/targz/GSM4509027/cell.A.txt.gz',
 '/home/liuc9/github/scMOCHA-data/data/GSE149689/targz/GSM4509027/cell.T.txt.gz',
 '/home/liuc9/github/scMOCHA-data/data/GSE149689/targz/GSM4509027/cell.C.txt.gz',
 '/home/liuc9/github/scMOCHA-data/data/GSE149689/targz/GSM4509027/cell.G.txt.gz']

In [35]:
letters = list("ATCG")

base_coverage_dict = load_mgatk_output(MGATK_OUT_DIR, mito_length, sample_prefix)
cell_barcodes = base_coverage_dict["A"][0].index


In [16]:
total_coverage = pd.DataFrame(np.zeros((len(cell_barcodes), mito_length)), index=cell_barcodes, columns=np.arange(1, mito_length + 1))
for nt in base_coverage_dict:
    total_coverage += base_coverage_dict[nt][0]
    total_coverage += base_coverage_dict[nt][1]


In [36]:
base_coverage_dict["T"][0].head()

,1,2,3,4,5,6,7,8,9,10,...,16560,16561,16562,16563,16564,16565,16566,16567,16568,16569
AAACCCACAAGAGCTG-1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0.0,0.0,0,0,0.0,0,0,0.0,0
AAACCCACACAGCCAC-1,0,0,2.0,0,0,0,0,0.0,0.0,3.0,...,0,0.0,4.0,0,0,0.0,0,0,3.0,0
AAACGAACAACGATCT-1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0.0,0.0,0,0,0.0,0,0,0.0,0
AAACGAACACCAATTG-1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0.0,0.0,0,0,0.0,0,0,0.0,0
AAACGAAGTACCGCGT-1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0.0,0.0,0,0,0.0,0,0,0.0,0


In [37]:
for nt in base_coverage_dict:
    print(nt, base_coverage_dict[nt][0].shape, base_coverage_dict[nt][1].shape)


G (2121, 16569) (2121, 16569)
T (2121, 16569) (2121, 16569)
A (2121, 16569) (2121, 16569)
C (2121, 16569) (2121, 16569)
